# 00 - Setup Verification

This notebook verifies that all dependencies are correctly installed.

In [2]:
import sys
print(f"Python: {sys.version}")

Python: 3.12.7 (main, Nov 13 2024, 20:33:58) [GCC 11.4.0]


In [7]:
# Core dependencies
import tvm
print(f"TVM: {tvm.__version__}")

import tvm_ffi
print(f"TVM_FFI: {tvm_ffi.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")

import gymnasium
print(f"Gymnasium: {gymnasium.__version__}")

import stable_baselines3
print(f"Stable-Baselines3: {stable_baselines3.__version__}")

import numpy as np
print(f"NumPy: {np.__version__}")

import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

TVM: 0.23.0
TVM_FFI: 0.1.9.dev1+gc78e8b4ee
PyTorch: 2.6.0+cu124
Gymnasium: 1.2.3
Stable-Baselines3: 2.7.1
NumPy: 2.4.2
Matplotlib: 3.10.8


In [ ]:
# Kernels package
from workloads import list_workloads, get_workload

print("Available workloads:", list_workloads())

Available workloads: ['conv2d_56', 'conv2d_bias_add_56', 'matmul_1024', 'matmul_256']


In [ ]:
# Verify TVM can compile and run a simple workload
mod: tvm.IRModule = get_workload("matmul_256")
print("Workload IRModule loaded successfully")
mod.show()

Workload IRModule loaded successfully


### Lower the TIR PrimFunc to executable code

with the help of TVM's VM (yes, TVM's name contains "virtual machine")


In [ ]:
# This single line performs a lot of operations
# It compiles the TIR module to executable code
built: tvm.runtime.module.Module = tvm.build(mod, target="llvm")
built.is_runnable()

True

In [ ]:
np.random.seed(42)
A_np: np.ndarray = np.random.randn(256, 256).astype("float32")
B_np: np.ndarray = np.random.randn(256, 256).astype("float32")
C_np: np.ndarray = np.empty(shape=(256, 256)).astype("float32")

# Pass the numpy arrays into TVM tensors through tvm_ffi interface
A_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(A_np)
B_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(B_np)
C_tvm: tvm.runtime._tensor.Tensor = tvm_ffi.from_dlpack(C_np)
# C is the output, passed by reference in the function

verif = A_np.dot(B_np)
built(A_tvm, B_tvm, C_tvm) # <- C = A dot B

## Verify numerical equality

In [30]:
np.allclose(C_tvm.numpy(), verif, rtol=1e-4, atol=1e-4)

# Expected "True"

True